# README
- **Author**: 詹明翰、謝子詮、曾翊涵
- **Created At**: 2026-11-26
- **Last Modified At**: 2026-11-26

## What does this file do?

Unit 2.1正式建立2024–2025第一當事人操作型候選事故資料，保留45,130筆。沿用Unit1.1的49列schema與研究ID規則，只從舊Notebook擷取靜態literal，不執行其程式。保留所有來源欄原字串，逐項建立有效值及三態特徵；無全欄刪除、無去重、無座標補值、無空間轉換／歸里、無里層級彙整或模型。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。以上為預定分工；本輪由Codex協助執行，成員實際貢獻、DA驗收及PM審查尚待確認，不以共同作者推定已完成工作。

Deliverables：一份正式事故CSV、一份schema／完整資料字典、處理及獨立資料說明Notebook。Actual Contribution：Codex本地執行，人工驗收待完成。

## What does this file take?

兩年度CSV為正式觀測輸入，事故PDF及已完成Notebook／Unit17 config只作固定版本規範證據。八項精確path／SHA如下：
- `data/raw/113年-臺北市死傷交通事故明細.csv` — `b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5`。
- `data/raw/114年-臺北市死傷交通事故明細.csv` — `61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c`。
- `data/raw/事故對照表.pdf` — `f40ded1ded1c816a38f421bc972cdc1320bbd280e647792207ec037d05fd3670`。
- `data/raw/taipei_2024_2025_crashes_raw_documentation.ipynb` — `0107387bd2362bd4c1a4db1056e5dbddcaad88cc362435d1236c1d7a352f4381`。
- `data/code/taipei_2024_2025_crash_validation.ipynb` — `3674d0043289f75079b46b90076795967563d721a91e303a568ecf85a25c0c6d`。
- `data/code/taipei_data_integration_readiness.ipynb` — `6048ff52d7653c9ec9d4b32cf2b78da2fb1b22dc0db0164cc48c4d2d98dd338b`。
- `data/code/taipei_supplementary_accident_boundary_validation.ipynb` — `f9df6bce7c6cdfcee95df55aa83de7187a3db3bcc76dfe76356699f6caab4dc6`。
- `data/code/unit17_validation_config.json` — `7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7`。

原始說明：[兩年事故文件](../raw/taipei_2024_2025_crashes_raw_documentation.ipynb)。研究規格依本次Unit2.1及[PROJECT](../../PROJECT.md)，不將可更新PROJECT當不變資料輸入。補充事故CSV／ZIP不讀取，不重新掃描全國資料或執行Chapter1。

## What does this file output?

- `data/processed/taipei_crashes_2024_2025_standardized.csv`：Role `analysis`，事故層級正式標準化資料；預期45,130列、研究ID唯一，供Unit2.2／2.3使用，不是最終GIS或分群樣本。[資料說明](../processed/taipei_crashes_2024_2025_standardized_documentation.ipynb)。
- `data/processed/taipei_crashes_2024_2025_standardized_schema.json`：Role `metadata`，完整欄序／dtype／單位／來源／品質與CSV讀取契約；不是第二份事故樣本。

UTF-8-SIG、LF換行、缺值`__MISSING__`、布林True／False；來源原始空字串保留為空，不與年度不適用的NA混同。讀取契約明示型態，不用CSV自動推論。輸出後獨立讀回逐值／dtype驗證；既有同bytes輸出可重新驗證，不同內容則停止，不覆寫未知版本。來源bytes不改。


In [1]:
from pathlib import Path
import sys,hashlib,json,csv,ast,io,platform
import pandas as pd
import numpy as np
import nbformat
from IPython.display import display,Markdown
for candidate in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
    if (candidate/'AGENTS.md').is_file() and (candidate/'templates').is_dir():
        PROJECT_ROOT=candidate;break
else:raise FileNotFoundError('請從研究工作包內執行')
assert Path(sys.executable).resolve()==(PROJECT_ROOT/'.venv/Scripts/python.exe').resolve()
TEMP=PROJECT_ROOT/'data/temp/unit21_validation';TEMP.mkdir(parents=True,exist_ok=True)
def project_file(relative,area=None,must_exist=True):
    rel=Path(relative)
    assert not rel.is_absolute() and '..' not in rel.parts
    path=(PROJECT_ROOT/rel).resolve()
    allowed=PROJECT_ROOT/'data'/area if area else PROJECT_ROOT/'data'
    assert path.is_relative_to(allowed.resolve())
    if must_exist:assert path.is_file(),relative
    return path
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
def show(title,frame,block=18):
    display(Markdown('**'+title+'**'))
    if not len(frame):print('無符合紀錄')
    for i in range(0,len(frame),block):display(frame.iloc[i:i+block])
pd.set_option('display.max_columns',None);pd.set_option('display.max_colwidth',110)
print('Python',platform.python_version(),'pandas',pd.__version__,'numpy',np.__version__,'既有.venv／DSSI Urban Crash；不安裝套件')


Python 3.12.2 pandas 3.1.1 numpy 2.5.5 既有.venv／DSSI Urban Crash；不安裝套件


## 1. 固定來源、邏輯列與觀察單位

所有八項必要輸入先比固定hash；不以當下hash重設expected。CSV以原字串讀取，csv.reader交叉驗證邏輯列號（不含header、1起算），不按清理後列序建ID。

In [2]:
EXPECTED_INPUTS={'data/raw/113年-臺北市死傷交通事故明細.csv': 'b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5', 'data/raw/114年-臺北市死傷交通事故明細.csv': '61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c', 'data/raw/事故對照表.pdf': 'f40ded1ded1c816a38f421bc972cdc1320bbd280e647792207ec037d05fd3670', 'data/raw/taipei_2024_2025_crashes_raw_documentation.ipynb': '0107387bd2362bd4c1a4db1056e5dbddcaad88cc362435d1236c1d7a352f4381', 'data/code/taipei_2024_2025_crash_validation.ipynb': '3674d0043289f75079b46b90076795967563d721a91e303a568ecf85a25c0c6d', 'data/code/taipei_data_integration_readiness.ipynb': '6048ff52d7653c9ec9d4b32cf2b78da2fb1b22dc0db0164cc48c4d2d98dd338b', 'data/code/taipei_supplementary_accident_boundary_validation.ipynb': 'f9df6bce7c6cdfcee95df55aa83de7187a3db3bcc76dfe76356699f6caab4dc6', 'data/code/unit17_validation_config.json': '7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7'}
for relative,expected in EXPECTED_INPUTS.items():assert sha(project_file(relative))==expected,relative
INPUTS={2024:'data/raw/113年-臺北市死傷交通事故明細.csv',2025:'data/raw/114年-臺北市死傷交通事故明細.csv'}
raw={};first={};manifest=[]
for year,relative in INPUTS.items():
    path=project_file(relative,'raw');f=pd.read_csv(path,encoding='utf-8-sig',dtype='string',keep_default_na=False)
    with path.open(encoding='utf-8-sig',newline='') as stream:logical=list(csv.reader(stream))
    assert len(logical)-1==len(f) and all(logical[1:])
    assert logical[0]==list(f)
    assert not f.eq('__MISSING__').any(axis=None)
    seq=pd.to_numeric(f['當事人序號'],errors='coerce');g=f[seq.eq(1)].copy()
    assert f.shape=={2024:(51810,49),2025:(22762,47)}[year]
    assert len(g)=={2024:22368,2025:22762}[year]
    if year==2025:assert seq.eq(1).all()
    raw[year]=f;first[year]=g
    manifest.append(dict(year=year,path=relative,bytes=path.stat().st_size,sha256=EXPECTED_INPUTS[relative],raw_party_rows=len(f),first_party_candidates=len(g),excluded_nonfirst_party=len(f)-len(g)))
show('輸入與唯一母體選取',pd.DataFrame(manifest))


**輸入與唯一母體選取**

,year,path,bytes,sha256,raw_party_rows,first_party_candidates,excluded_nonfirst_party
0,2024,data/raw/113年-臺北市死傷交通事故明細.csv,9398043,b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5,51810,22368,29442
1,2025,data/raw/114年-臺北市死傷交通事故明細.csv,4082763,61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c,22762,22762,0


## 2. 沿用49列欄位對應及碼表證據

道路照明與光線拆為兩欄並加適用性標記；F08採中性fatalities_source_count，否定舊暫定24h欄名的正式認證解讀。所有來源文字含前導0保留，標準類別碼為字串；完整／部分碼表的未列碼語義分開。

In [3]:
legacy=nbformat.read(project_file('data/code/taipei_2024_2025_crash_validation.ipynb'),as_version=4)
def literal(name):
    source=next(c.source for c in legacy.cells if c.cell_type=='code' and c.source.startswith(name+' ='))
    return ast.literal_eval(ast.parse(source).body[0].value)
SCHEMA=literal('SCHEMA');BOOKS=literal('BOOKS');assert len(SCHEMA)==49
SPECS=[]
for row in SCHEMA:
    if row['field_id']=='F14':
        for year,original,name in [(2024,'道路照明設備','road_lighting_2024_code'),(2025,'光線','light_condition_2025_code')]:
            SPECS.append(dict(row,normalized_name=name,source_columns={year:original},definition='僅'+str(year)+'來源'+original+'；不認證與另一年語義等價'))
    else:
        name={'F08':'fatalities_source_count','F09':'fatalities_2_30d_count','F48':'coordinate_x','F49':'coordinate_y'}.get(row['field_id'],row['建議英文名'])
        definition=row['中文定義']
        if row['field_id']=='F08':definition='原死亡人數：時間窗口尚未完整確認，中性名稱；不與2–30日死亡合併'
        SPECS.append(dict(row,normalized_name=name,source_columns={y:row[str(y)+'原始欄'] for y in [2024,2025] if row[str(y)+'原始欄']!='不存在'},definition=definition))
assert len(SPECS)==50
for year in INPUTS:assert {v for s in SPECS for y,v in s['source_columns'].items() if y==year}==set(raw[year])
show('49列對應實作為50個分開的來源變數',pd.DataFrame([{'field_id':s['field_id'],'standard_name':s['normalized_name'],'source_columns':str(s['source_columns']),'definition':s['definition']} for s in SPECS]))


**49列對應實作為50個分開的來源變數**

,field_id,standard_name,source_columns,definition
0,F01,accident_year,"{2024: '發生年度', 2025: '發生年度'}",事故發生年，檔內為民國 113／114；對應西元時在記憶體加 1911
1,F02,accident_month,"{2024: '發生月', 2025: '發生月'}",事故發生月份
2,F03,accident_day,"{2024: '發生日', 2025: '發生日'}",事故發生日
3,F04,accident_hour,"{2024: '發生時-Hours', 2025: '發生時-Hours'}",事故發生小時；檔內 24 小時制數值
4,F05,accident_minute,"{2024: '發生分', 2025: '發生分'}",事故發生分鐘
5,F06,district_label_source,"{2024: '區序', 2025: '區序'}",本機字串含行政區序號及區名；正式區序代碼規格未知
6,F07,accident_location_text,"{2024: '肇事地點', 2025: '肇事地點'}",事故地點文字；不是當事人居住地址
7,F08,fatalities_source_count,"{2024: '死亡人數', 2025: '死亡人數'}",原死亡人數：時間窗口尚未完整確認，中性名稱；不與2–30日死亡合併
8,F09,fatalities_2_30d_count,"{2024: '2-30日死亡人數', 2025: '2-30日死亡人數'}",欄名表示事故 2–30 日死亡人數；彙整、追蹤及發布缺值口徑待官方確認
9,F10,injured_count,"{2024: '受傷人數', 2025: '受傷人數'}",事故受傷人數；與單一當事人的受傷程度不同


,field_id,standard_name,source_columns,definition
18,F18,accident_position_code,"{2024: '事故位置', 2025: '事故位置'}",事故發生位置類別
19,F19,surface_material_code,"{2024: '路面狀況1', 2025: '路面狀況1'}",路面鋪裝代碼
20,F20,surface_condition_code,"{2024: '路面狀況2', 2025: '路面狀況2'}",路面狀態代碼
21,F21,surface_defect_code,"{2024: '路面狀況3', 2025: '路面狀況3'}",路面缺陷代碼
22,F22,road_obstacle_code,"{2024: '道路障礙1', 2025: '道路障礙1'}",障礙物代碼
23,F23,sight_condition_code,"{2024: '道路障礙2', 2025: '道路障礙2'}",視距代碼；新版及表單內舊碼不可混用
24,F24,signal_type_code,"{2024: '號誌1', 2025: '號誌1'}",號誌種類代碼
25,F25,signal_status_code,"{2024: '號誌2', 2025: '號誌2'}",號誌動作代碼
26,F26,direction_divider_code,"{2024: '車道劃分-分向', 2025: '車道劃分-分向'}",分向設施代碼
27,F27,general_lane_divider_code,"{2024: '車道劃分-分道1', 2025: '車道劃分-分道1'}",快車道或一般車道間分道設施代碼


,field_id,standard_name,source_columns,definition
36,F36,first_party_phone_use_code,"{2024: '行動電話', 2025: '行動電話'}",行動電話、電腦或類似裝置使用情形
37,F37,first_party_vehicle_use_code,"{2024: '車輛用途', 2025: '車輛用途'}",車輛用途／非駕駛人類別
38,F38,first_party_action_code,"{2024: '當事者行動狀態', 2025: '當事者行動狀態'}",車或人的事故當時行動狀態
39,F39,first_party_license_status_code,"{2024: '駕駛資格情形', 2025: '駕駛資格情形'}",駕照狀態代碼
40,F40,first_party_license_type_code,"{2024: '駕駛執照種類', 2025: '駕駛執照種類'}",駕照種類代碼
41,F41,first_party_alcohol_code,"{2024: '飲酒情形', 2025: '飲酒情形'}",酒精觀察／檢測分類，不是連續酒測值
42,F42,first_party_initial_impact_code,"{2024: '車輛撞擊部位1', 2025: '車輛撞擊部位1'}",最初撞擊部位候選對應
43,F43,first_party_other_impact_code,{2024: '車輛撞擊部位2'},其他撞擊部位候選對應；僅 2024 欄
44,F44,first_party_preliminary_cause_code,"{2024: '肇因碼-個別', 2025: '肇因碼-個別'}",初步分析研判之個別參照碼，不是法院責任或因果效應
45,F45,main_preliminary_cause_code,{2024: '肇因碼-主要'},主要初步分析研判候選欄；僅 2024 且全缺失


## 3. 正式標準化與逐變數有效性

原欄全部另存字串。數值結構與欄位語義分開；類別碼是字串、原前導0仍可回查。速限0／502退出預設有效速限但不刪事故；負年齡不納結構有效年齡，0及>100保留待審。傷亡獨立非負整數、不用個人傷勢代替總量。

In [4]:
def number(values):
    # Python float parses source decimals; original lexical values always retained.
    def parse(v):
        if pd.isna(v) or not str(v).strip():return pd.NA
        try:
            x=float(v);return x if np.isfinite(x) else pd.NA
        except (ValueError,TypeError,OverflowError):return pd.NA
    return values.map(parse).astype('Float64')
def nonnegative_integer(v):
    x=number(v);return (x.notna()&x.ge(0)&x.mod(1).eq(0)).fillna(False)
def integer(v):
    x=number(v);return x.where(x.notna()&x.mod(1).eq(0)).astype('Int64')
def numeric_code(v):
    return integer(v).astype('string')
def flag(condition,eligible):
    return condition.astype('boolean').where(eligible.astype('boolean'),pd.NA)

DICTIONARY={};frames=[]
for year,f in first.items():
    N=len(f);data={}
    def add(name,value,dtype,definition,unit='not applicable',source='',level='事故層級／診斷',missing='NA保留，不以0代填',basis='Unit2.1明示工作規格'):
        if not isinstance(value,pd.Series):value=pd.Series(value if isinstance(value,(list,np.ndarray)) else [value]*N)
        assert len(value)==N,name
        data[name]=value.reset_index(drop=True).astype(dtype)
        record=dict(name=name,dtype=dtype,definition=definition,unit=unit,source=source,observation_level=level,missing_rule=missing,basis=basis)
        if name in DICTIONARY:assert DICTIONARY[name]==record,name
        else:DICTIONARY[name]=record
    path=INPUTS[year];rows=f.index.to_numpy()+1
    add('record_id',[f'tpe-crash:{year}:{EXPECTED_INPUTS[path]}:r{i:06d}' for i in rows],'string','研究ID＝來源年＋原檔SHA256＋原CSV邏輯列；不是官方事故ID',source='source_year/source_file_sha256/source_row_number')
    add('source_year',year,'Int64','來源檔對應西元年',unit='year',source='來源版本')
    add('source_file',path,'string','精確原始CSV相對路徑',source='固定輸入清單')
    add('source_file_sha256',EXPECTED_INPUTS[path],'string','原始CSV完整SHA256',source='固定輸入清單')
    add('source_row_number',rows,'Int64','原CSV邏輯資料列，1起算，不含表頭；非物理行號',unit='row',source='原CSV邏輯列')
    for spec in SPECS:
        name=spec['normalized_name'];col=spec['source_columns'].get(year)
        v=f[col] if col else pd.Series(pd.NA,index=f.index,dtype='string')
        source=json.dumps(spec['source_columns'],ensure_ascii=False,sort_keys=True)
        add(name+'_raw',v,'string','原字串：'+spec['definition'],source=source,level=spec['觀察層級'],missing='原始空字串不改；年度無此欄為NA；由適用旗標區分',basis=spec['定義依據及未知'])
        if len(spec['source_columns'])==1:
            add(name+'_applicable',bool(col),'boolean','來源年度是否包含此欄；False為不適用，不是來源缺失',source=source)
        if name in ['accident_year','accident_month','accident_day','accident_hour','accident_minute','coordinate_x','coordinate_y']:continue
        if name in ['district_label_source','accident_location_text']:
            normalized=v.where(v.str.strip().ne(''));dtype='string'
        elif name in ['fatalities_source_count','fatalities_2_30d_count','injured_count','party_sequence']:
            normalized=integer(v).where(nonnegative_integer(v));dtype='Int64'
        elif name in ['speed_limit_source','first_party_age_source']:
            normalized=number(v);dtype='Float64'
        else:
            normalized=v.str.strip().where(v.str.strip().ne('')) if name=='first_party_vehicle_code' else numeric_code(v);dtype='string'
        add(name,normalized,dtype,spec['definition'],source=source,level=spec['觀察層級'],basis=spec['定義依據及未知'])
        book=next((b for b in BOOKS.values() if col and b['columns'].get(year)==col),None)
        # Annual unavailable fields still have the same reference evidence family.
        if book is None:
            book=next((b for b in BOOKS.values() if any(b['columns'].get(y)==c for y,c in spec['source_columns'].items())),None)
        if book:
            keys=[str(k) for k in book['labels']]
            known=normalized.isin(keys)
            add(name+'_reference_code_known',known,'boolean','是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤',source=source,basis='Unit1.1靜態碼表；完整性與未知見schema reference_codebooks')
    # Dates retain original ROC-year text above, while standard dates use CE.
    yr=integer(f['發生年度']);mo=integer(f['發生月']);dy=integer(f['發生日']);hr=integer(f['發生時-Hours']);mi=integer(f['發生分'])
    date=pd.to_datetime(pd.DataFrame({'year':yr+1911,'month':mo,'day':dy}),errors='coerce')
    date_ok=(date.notna()&yr.eq(year-1911)&mo.between(1,12)&dy.between(1,31)).fillna(False)
    time_ok=(hr.between(0,23)&mi.between(0,59)).fillna(False)
    for name,value,definition,unit in [('year',yr+1911,'西元事故年＝原民國年+1911','year'),('month',mo,'事故月份','month'),('day',dy,'事故日','day'),('hour',hr,'事故小時，0–23','hour'),('minute',mi,'事故分鐘，0–59','minute')]:
        add(name,value,'Int64',definition,unit=unit,source='原發生年／月／日／時／分')
    add('date_valid',date_ok,'boolean','合法曆日且原年符合來源年；獨立有效條件',source='年月日原欄')
    add('time_valid',time_ok,'boolean','整數小時0–23、分鐘0–59；獨立有效條件',source='時分原欄')
    add('accident_date',date.dt.strftime('%Y-%m-%d').where(date_ok),'string','ISO日期YYYY-MM-DD；臺灣當地日期',unit='date',source='年月日原欄')
    time=hr.astype('string').str.zfill(2)+':'+mi.astype('string').str.zfill(2)
    add('accident_time',time.where(time_ok),'string','HH:MM；原資料分鐘精度，不虛構秒數',unit='local minute',source='時分原欄')
    dt=date.dt.strftime('%Y-%m-%d')+'T'+time+'+08:00'
    add('accident_datetime',dt.where(date_ok&time_ok),'string','ISO分鐘時間YYYY-MM-DDTHH:MM+08:00；Asia/Taipei研究慣例',unit='local datetime',source='年月日時分原欄')
    weekday=date.dt.dayofweek.astype('Int64').where(date_ok)
    add('day_of_week',weekday,'Int64','星期一0至星期日6，不等同國定假日',unit='weekday',source='合法日期')
    n=number(f['受傷人數']);death=number(f['死亡人數']);late=number(f['2-30日死亡人數'])
    valid_type=numeric_code(f['事故類型及型態']).isin([str(k) for k in BOOKS['accident_type']['labels']])
    valid_position=numeric_code(f['事故位置']).isin([str(k) for k in BOOKS['position']['labels']])
    for name,valid,source in [('accident_type_valid',valid_type,'事故類型及型態'),('accident_position_valid',valid_position,'事故位置'),('injured_count_structural_valid',nonnegative_integer(f['受傷人數']),'受傷人數'),('fatalities_source_count_structural_valid',nonnegative_integer(f['死亡人數']),'死亡人數'),('fatalities_2_30d_count_structural_valid',nonnegative_integer(f['2-30日死亡人數']),'2-30日死亡人數')]:
        add(name,valid,'boolean','該欄獨立結構／印列碼有效性；不認證死亡時間窗或全部語義',source=source)
    speed=number(f['速限-速度限制']);speed_ok=nonnegative_integer(f['速限-速度限制'])&speed.gt(0)&speed.ne(502)
    add('speed_limit_valid',speed.where(speed_ok),'Float64','預設速限補充數值：有限非負整數且排除0及502，不是實際車速',unit='來源速限；km/h工作解讀，完整發布規格未認證',source='速限-速度限制')
    for name,condition,definition in [('speed_zero_flag',speed.eq(0),'速限原值0，意義待確認'),('speed_502_flag',speed.eq(502),'速限原值502，不自動改50')]:add(name,condition.fillna(False),'boolean',definition,source='速限-速度限制')
    age=number(f['年齡']);age_ok=nonnegative_integer(f['年齡'])
    add('first_party_age_valid_structural',age.where(age_ok),'Float64','僅有限非負整數年齡；0及>100仍保留但待審，不認證為有效自然人年齡',unit='歲（來源年齡）',source='年齡',level='第一當事人')
    for name,condition,definition in [('age_negative_flag',age.lt(0),'負年齡排除於結構有效年齡'),('age_zero_review_flag',age.eq(0),'0歲僅待確認，不直接判錯'),('age_above_100_review_flag',age.gt(100),'超100歲僅待確認，不截尾')]:add(name,condition.fillna(False),'boolean',definition,source='年齡',level='第一當事人')
    x=number(f['座標-X']);y=number(f['座標-Y']);parse=x.notna()&y.notna();missing=f['座標-X'].str.strip().eq('')|f['座標-Y'].str.strip().eq('')
    add('coordinate_x_raw',f['座標-X'],'string','原X字串完整保留；與coordinate_x_raw來源映射對應',source='座標-X') if 'coordinate_x_raw' not in data else None
    add('coordinate_y_raw',f['座標-Y'],'string','原Y字串完整保留',source='座標-Y') if 'coordinate_y_raw' not in data else None
    add('longitude_candidate',x,'Float64','X之經度工作解讀；datum未正式認證，採EPSG4326工作慣例',unit='degree candidate',source='座標-X')
    add('latitude_candidate',y,'Float64','Y之緯度工作解讀；datum未正式認證，無CRS轉換或geometry',unit='degree candidate',source='座標-Y')
    add('coordinate_parse_valid',parse,'boolean','X及Y皆可解析為有限數；不是里內或CRS認證',source='座標-X／Y')
    add('coordinate_missing_flag',missing,'boolean','任一原座標空白；不補值',source='座標-X／Y')
    add('coordinate_global_range_valid',(parse&x.between(-180,180)&y.between(-90,90)).fillna(False),'boolean','候選經緯度全球度數範圍；不是臺北位置認證',source='座標-X／Y')
    minutes=hr*60+mi;type_code=integer(f['事故類型及型態']);pos=integer(f['事故位置'])
    definitions={
        'morning_peak':(time_ok,minutes.ge(420)&minutes.lt(540),'07:00含至09:00不含'),
        'evening_peak':(time_ok,minutes.ge(1020)&minutes.lt(1140),'17:00含至19:00不含'),
        'late_night':(time_ok,minutes.ge(1320)|minutes.lt(360),'22:00含或06:00之前'),
        'weekend':(date_ok,weekday.isin([5,6]),'星期六或日；非所有國定假日'),
        'intersection':(valid_position,pos.isin([1,2,3,4]),'位置1–4：路口及附近／待轉／停等'),
        'rear_end':(valid_type,type_code.eq(13),'型態13追撞'),
        'side_collision':(valid_type,type_code.eq(16),'型態16側撞，不混同擦撞'),
        'pedestrian_collision':(valid_type,type_code.between(1,9),'型態1–9人與車代理，不等全部行人涉入'),
        'multi_injury':(nonnegative_integer(f['受傷人數']),n.ge(2),'事故受傷人數至少2；不是重傷程度'),
        'intersection_strict':(valid_position,pos.eq(1),'額外嚴格路口：位置1；有效分母同intersection')}
    for name,(eligible,condition,definition) in definitions.items():
        add(name,flag(condition,eligible),'boolean',definition+'；True有效符合／False有效不符／NA不可判定',source='各自時間／日期／型態／位置／受傷欄',missing='有效性False則特徵NA；不得當False')
        add(name+'_valid',eligible,'boolean','此特徵獨立有效分母資格；與其他特徵缺失無關',source=name+'之定義欄')
    sev=integer(f['受傷程度']);sev_known=sev.isin([1,2,3,4,5])
    r20_valid=sev_known&((sev.eq(1)&nonnegative_integer(f['死亡人數']))|(sev.eq(2)&nonnegative_integer(f['受傷人數']))|(sev.eq(5)&nonnegative_integer(f['2-30日死亡人數']))|sev.isin([3,4]))
    suspicious=(sev.eq(1)&death.eq(0))|(sev.eq(5)&late.eq(0))|(sev.eq(2)&n.eq(0))
    add('conditional_casualty_inconsistency_flag',flag(suspicious,r20_valid),'boolean','R20條件疑點：程度1且原死亡0，或程度5且2–30日死亡0，或程度2且受傷0；不判哪欄錯誤',source='受傷程度／三個事故傷亡原欄',missing='相應欄不可判則NA；不因此改multi_injury分母')
    add('r20_check_valid',r20_valid,'boolean','R20對應欄可判讀；不是傷亡語義認證',source='受傷程度及相應總量')
    key=['發生年度','發生月','發生日','發生時-Hours','發生分','肇事地點','座標-X','座標-Y']
    add('candidate_event_key_duplicate_flag',f.duplicated(key,keep=False),'boolean','候選時地座標鍵重複，包含空白；不合併或刪除，非官方事件ID',source='原年月日時分／地點／X／Y')
    add('time_address_key_duplicate_flag',f.duplicated(key[:-2],keep=False),'boolean','只以日時分及地址的非唯一性；不拿座標作跨來源驗證前提',source='原年月日時分／地點')
    add('raw_record_duplicate_flag',f.duplicated(keep=False),'boolean','第一當事人完整原始列重複；標記而不刪除',source='全部適用原欄')
    add('raw_any_blank_flag',f.apply(lambda c:c.str.strip().eq('')).any(axis=1),'boolean','至少一個該年原始欄空白；年度不適用欄不混入，禁止全欄刪除',source='該年適用的全部原欄')
    frames.append(pd.DataFrame(data))
df=pd.concat(frames,ignore_index=True)
assert len(df)==45130 and df.record_id.is_unique and df.record_id.notna().all()
assert all('24h' not in c for c in df)
assert 'geometry' not in df and 'village_id' not in df
assert list(df)==list(DICTIONARY)
show('正式事故資料年度規模',df.groupby('source_year').size().rename('N').reset_index())
print('總列數／欄位數',df.shape,'未刪除缺失、座標或空間異常事故')


C:\Users\user\OneDrive\桌面\大學\課程\院學士\資料科學與社會研究\DSSI2026-group-2\econ-5166-student-kit\data\temp\unit21_validation\kernel_temp\ipykernel_26492\4231353052.py:73: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekday=date.dt.dayofweek.astype('Int64').where(date_ok)


C:\Users\user\OneDrive\桌面\大學\課程\院學士\資料科學與社會研究\DSSI2026-group-2\econ-5166-student-kit\data\temp\unit21_validation\kernel_temp\ipykernel_26492\4231353052.py:73: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekday=date.dt.dayofweek.astype('Int64').where(date_ok)


**正式事故資料年度規模**

,source_year,N
0,2024,22368
1,2025,22762


總列數／欄位數 (45130, 191) 未刪除缺失、座標或空間異常事故


## 4. 品質檢查、三態特徵及完整來源回查

每項特徵使用自己的有效性，不以全欄完整案例刪除。時段以分鐘端點測試，日期星期另按Python datetime獨立核對；所有適用原欄及來源邏輯列逐值比對。

In [5]:
FEATURES=['morning_peak','evening_peak','late_night','weekend','intersection','rear_end','side_collision','pedestrian_collision','multi_injury']
feature_counts=[];quality=[]
for year,g in df.groupby('source_year',sort=True):
    source=first[int(year)].reset_index(drop=True)
    assert g.source_row_number.tolist()==(first[int(year)].index+1).tolist()
    expected_ids=[f'tpe-crash:{year}:{EXPECTED_INPUTS[INPUTS[int(year)]]}:r{i:06d}' for i in g.source_row_number]
    assert g.record_id.tolist()==expected_ids
    for spec in SPECS:
        col=spec['source_columns'].get(int(year));name=spec['normalized_name']+'_raw'
        if col:pd.testing.assert_series_equal(g[name].reset_index(drop=True),source[col],check_names=False)
        else:assert g[name].isna().all()
    from datetime import datetime
    parsed=pd.to_datetime(g.accident_datetime,format='%Y-%m-%dT%H:%M%z',errors='raise')
    assert parsed.dt.year.eq(year).all() and parsed.dt.tz_convert('Asia/Taipei').dt.hour.eq(g.hour).all()
    assert g.day_of_week.tolist()==[datetime.strptime(x,'%Y-%m-%d').weekday() for x in g.accident_date]
    assert g.date_valid.all() and g.time_valid.all()
    expected={'accident_type_valid':{2024:22359,2025:22750},'accident_position_valid':{2024:22359,2025:22750},'injured_count_structural_valid':{2024:22367,2025:22761},'fatalities_source_count_structural_valid':{2024:22359,2025:22755},'fatalities_2_30d_count_structural_valid':{2024:22359,2025:22752},'coordinate_parse_valid':{2024:22367,2025:22760}}
    for column,counts in expected.items():assert int(g[column].sum())==counts[int(year)],(column,year)
    assert int(g.coordinate_missing_flag.sum())=={2024:1,2025:2}[int(year)]
    assert int(g.conditional_casualty_inconsistency_flag.sum())=={2024:6,2025:4}[int(year)]
    assert g.loc[g.speed_zero_flag|g.speed_502_flag,'speed_limit_valid'].isna().all()
    assert g.loc[g.age_negative_flag,'first_party_age_valid_structural'].isna().all()
    for feature in FEATURES:
        assert g[feature].notna().equals(g[feature+'_valid'].astype(bool))
        feature_counts.append(dict(year=int(year),feature=feature,True_N=int(g[feature].eq(True).sum()),False_N=int(g[feature].eq(False).sum()),NA_N=int(g[feature].isna().sum()),valid_N=int(g[feature+'_valid'].sum()),candidate_N=len(g)))
    quality.append(dict(year=int(year),N=len(g),date_time_valid=int((g.date_valid&g.time_valid).sum()),missing_coordinates=int(g.coordinate_missing_flag.sum()),R20=int(g.conditional_casualty_inconsistency_flag.sum()),R20_unknown=int(g.conditional_casualty_inconsistency_flag.isna().sum()),speed_0=int(g.speed_zero_flag.sum()),speed_502=int(g.speed_502_flag.sum()),age_negative=int(g.age_negative_flag.sum()),age_zero=int(g.age_zero_review_flag.sum()),age_over100=int(g.age_above_100_review_flag.sum()),duplicate_source_rows=int(g.raw_record_duplicate_flag.sum()),candidate_key_duplicate_rows=int(g.candidate_event_key_duplicate_flag.sum()),time_address_duplicate_rows=int(g.time_address_key_duplicate_flag.sum())))
# Endpoint expectations explicitly differ at excluded/ included bounds.
boundaries=pd.Series([0,359,360,419,420,539,540,1019,1020,1139,1140,1319,1320,1439])
assert (boundaries.ge(420)&boundaries.lt(540)).tolist()==[False,False,False,False,True,True,False,False,False,False,False,False,False,False]
assert (boundaries.ge(1020)&boundaries.lt(1140)).tolist()==[False,False,False,False,False,False,False,False,True,True,False,False,False,False]
assert (boundaries.ge(1320)|boundaries.lt(360)).tolist()==[True,True,False,False,False,False,False,False,False,False,False,False,True,True]
unknown=flag(pd.Series([True,False,True]),pd.Series([True,True,False]));assert unknown.tolist()[:2]==[True,False] and pd.isna(unknown.iloc[2])
show('九項特徵各年度True／False／NA及有效分母',pd.DataFrame(feature_counts))
show('品質及特殊值；均不刪事故',pd.DataFrame(quality))
show('R20十筆原值與中性分析值',df.loc[df.conditional_casualty_inconsistency_flag.fillna(False),['record_id','source_year','source_row_number','first_party_injury_severity_code_raw','fatalities_source_count_raw','fatalities_2_30d_count_raw','injured_count_raw','conditional_casualty_inconsistency_flag']])
show('三筆缺座標完整保留',df.loc[df.coordinate_missing_flag,['record_id','source_year','source_row_number','accident_location_text','coordinate_x_raw','coordinate_y_raw','longitude_candidate','latitude_candidate','coordinate_missing_flag']])


**九項特徵各年度True／False／NA及有效分母**

,year,feature,True_N,False_N,NA_N,valid_N,candidate_N
0,2024,morning_peak,3323,19045,0,22368,22368
1,2024,evening_peak,3332,19036,0,22368,22368
2,2024,late_night,2119,20249,0,22368,22368
3,2024,weekend,4988,17380,0,22368,22368
4,2024,intersection,13556,8803,9,22359,22368
5,2024,rear_end,3131,19228,9,22359,22368
6,2024,side_collision,4483,17876,9,22359,22368
7,2024,pedestrian_collision,1969,20390,9,22359,22368
8,2024,multi_injury,6253,16114,1,22367,22368
9,2025,morning_peak,3415,19347,0,22762,22762


**品質及特殊值；均不刪事故**

,year,N,date_time_valid,missing_coordinates,R20,R20_unknown,speed_0,speed_502,age_negative,age_zero,age_over100,duplicate_source_rows,candidate_key_duplicate_rows,time_address_duplicate_rows
0,2024,22368,22368,1,6,227,328,1,1,222,11,0,0,22
1,2025,22762,22762,2,4,215,315,0,0,217,18,0,0,22


**R20十筆原值與中性分析值**

,record_id,source_year,source_row_number,first_party_injury_severity_code_raw,fatalities_source_count_raw,fatalities_2_30d_count_raw,injured_count_raw,conditional_casualty_inconsistency_flag
2972,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r006876,2024,6876,5,0,0,1,True
3986,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r009212,2024,9212,5,0,0,1,True
8529,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r019829,2024,19829,5,0,0,3,True
17457,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r040419,2024,40419,5,0,0,2,True
17498,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r040515,2024,40515,5,0,0,1,True
19983,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r046256,2024,46256,5,0,0,1,True
30652,tpe-crash:2025:61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c:r008285,2025,8285,5,0,0,2,True
31364,tpe-crash:2025:61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c:r008997,2025,8997,5,0,0,1,True
35851,tpe-crash:2025:61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c:r013484,2025,13484,5,0,0,2,True
42882,tpe-crash:2025:61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c:r020515,2025,20515,5,0,0,2,True


**三筆缺座標完整保留**

,record_id,source_year,source_row_number,accident_location_text,coordinate_x_raw,coordinate_y_raw,longitude_candidate,latitude_candidate,coordinate_missing_flag
16594,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r038470,2024,38470,文山區興隆路4段66號前,,,<NA>,<NA>,True
26801,tpe-crash:2025:61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c:r004434,2025,4434,大同區太原路50號,,,<NA>,<NA>,True
43928,tpe-crash:2025:61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c:r021561,2025,21561,大安區辛亥路3段與芳蘭路口國三甲聯絡道,,,<NA>,<NA>,True


## 5. 完整資料字典與CSV契約

原字串空白仍為空字串；衍生缺值及來源年度無此欄為__MISSING__，配合_applicable旗標辨識。程式讀取先全字串，再依schema明轉Int64／Float64／boolean，避免CSV自動推論損失ID、前導0或NA。

In [6]:
for name,record in DICTIONARY.items():
    if name in ['fatalities_source_count','fatalities_2_30d_count','injured_count']:record['unit']='人；結構有效總量，時間／追蹤語義依欄限制'
    elif name=='party_sequence':record['unit']='當事人序次，非肇責'
    elif name=='first_party_age_source':record['unit']='歲（來源數值）'
    elif name=='speed_limit_source':record['unit']='來源速限；km/h工作解讀，非實際速度'
    elif record['dtype']=='boolean':record['unit']='logical flag；True/False/NA依欄定義'
    elif name.endswith('_code'):record['unit']='類別碼字串，無數值大小尺度'
contract={'unit':'2.1','created_at':'2026-11-26','dataset_path':'data/processed/taipei_crashes_2024_2025_standardized.csv','producer':'data/code/prepare_taipei_crashes_2024_2025.ipynb','documentation':'data/processed/taipei_crashes_2024_2025_standardized_documentation.ipynb','role':'analysis','observation_unit':'first-party operational candidate accident, not certified official unique event','expected_rows':45130,'expected_year_counts':{'2024':22368,'2025':22762},'csv':{'encoding':'utf-8-sig','line_terminator':'LF','missing_sentinel':'__MISSING__','original_empty_strings':'preserved; keep_default_na=False','boolean_values':['True','False'],'date_format':'%Y-%m-%d','time_format':'%H:%M','datetime_format':'%Y-%m-%dT%H:%M%z','timezone':'Asia/Taipei; serialized +08:00'},'source_inputs':EXPECTED_INPUTS,'columns':list(DICTIONARY.values()),'schema_mapping_49':SCHEMA,'implemented_source_variables_50':SPECS,'reference_codebooks':BOOKS,'features':FEATURES,'restrictions':['no spatial join/geometry/village_id','no coordinate imputation','no raw overwrite or deduplication','fatalities_source_count time window unconfirmed','R20 and cross-source differences not resolved','first-party demographics not all parties or residents','not final clustering input; independent valid denominators','light and lighting annual fields are not merged']}
dictionary=pd.DataFrame(contract['columns'])
show('完整欄位字典：型態／單位／層級／來源／缺值',dictionary,block=12)


**完整欄位字典：型態／單位／層級／來源／缺值**

,name,dtype,definition,unit,source,observation_level,missing_rule,basis
0,record_id,string,研究ID＝來源年＋原檔SHA256＋原CSV邏輯列；不是官方事故ID,not applicable,source_year/source_file_sha256/source_row_number,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
1,source_year,Int64,來源檔對應西元年,year,來源版本,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
2,source_file,string,精確原始CSV相對路徑,not applicable,固定輸入清單,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
3,source_file_sha256,string,原始CSV完整SHA256,not applicable,固定輸入清單,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
4,source_row_number,Int64,原CSV邏輯資料列，1起算，不含表頭；非物理行號,row,原CSV邏輯列,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
5,accident_year_raw,string,原字串：事故發生年，檔內為民國 113／114；對應西元時在記憶體加 1911,not applicable,"{""2024"": ""發生年度"", ""2025"": ""發生年度""}",事故時間,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 發生時間；原始值與檔名核對
6,accident_month_raw,string,原字串：事故發生月份,not applicable,"{""2024"": ""發生月"", ""2025"": ""發生月""}",事故時間,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 發生時間
7,accident_day_raw,string,原字串：事故發生日,not applicable,"{""2024"": ""發生日"", ""2025"": ""發生日""}",事故時間,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 發生時間
8,accident_hour_raw,string,原字串：事故發生小時；檔內 24 小時制數值,not applicable,"{""2024"": ""發生時-Hours"", ""2025"": ""發生時-Hours""}",事故時間,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 發生時間；檔內有效性檢查
9,accident_minute_raw,string,原字串：事故發生分鐘,not applicable,"{""2024"": ""發生分"", ""2025"": ""發生分""}",事故時間,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 發生時間


,name,dtype,definition,unit,source,observation_level,missing_rule,basis
12,accident_location_text_raw,string,原字串：事故地點文字；不是當事人居住地址,not applicable,"{""2024"": ""肇事地點"", ""2025"": ""肇事地點""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 發生地點；發布 CSV 轉錄細節未知
13,accident_location_text,string,事故地點文字；不是當事人居住地址,not applicable,"{""2024"": ""肇事地點"", ""2025"": ""肇事地點""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 發生地點；發布 CSV 轉錄細節未知
14,fatalities_source_count_raw,string,原字串：原死亡人數：時間窗口尚未完整確認，中性名稱；不與2–30日死亡合併,not applicable,"{""2024"": ""死亡人數"", ""2025"": ""死亡人數""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 死傷人數；與官方頁 24 小時定義對照
15,fatalities_source_count,Int64,原死亡人數：時間窗口尚未完整確認，中性名稱；不與2–30日死亡合併,人；結構有效總量，時間／追蹤語義依欄限制,"{""2024"": ""死亡人數"", ""2025"": ""死亡人數""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 死傷人數；與官方頁 24 小時定義對照
16,fatalities_2_30d_count_raw,string,原字串：欄名表示事故 2–30 日死亡人數；彙整、追蹤及發布缺值口徑待官方確認,not applicable,"{""2024"": ""2-30日死亡人數"", ""2025"": ""2-30日死亡人數""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,CSV 欄名；PDF p.3 有個人 2–30 日死亡代碼，不能據此認證總數產製方式
17,fatalities_2_30d_count,Int64,欄名表示事故 2–30 日死亡人數；彙整、追蹤及發布缺值口徑待官方確認,人；結構有效總量，時間／追蹤語義依欄限制,"{""2024"": ""2-30日死亡人數"", ""2025"": ""2-30日死亡人數""}",事故情境／總量層級（候選）,NA保留，不以0代填,CSV 欄名；PDF p.3 有個人 2–30 日死亡代碼，不能據此認證總數產製方式
18,injured_count_raw,string,原字串：事故受傷人數；與單一當事人的受傷程度不同,not applicable,"{""2024"": ""受傷人數"", ""2025"": ""受傷人數""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 死傷人數
19,injured_count,Int64,事故受傷人數；與單一當事人的受傷程度不同,人；結構有效總量，時間／追蹤語義依欄限制,"{""2024"": ""受傷人數"", ""2025"": ""受傷人數""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 死傷人數
20,party_sequence_raw,string,原字串：事故內當事人序次的候選欄；1 為第一當事人紀錄,not applicable,"{""2024"": ""當事人序號"", ""2025"": ""當事人序號""}",第一當事人層級,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.3 第幾當事者；不是事故官方 ID，也不等同肇責認定
21,party_sequence,Int64,事故內當事人序次的候選欄；1 為第一當事人紀錄,當事人序次，非肇責,"{""2024"": ""當事人序號"", ""2025"": ""當事人序號""}",第一當事人層級,NA保留，不以0代填,PDF p.3 第幾當事者；不是事故官方 ID，也不等同肇責認定


,name,dtype,definition,unit,source,observation_level,missing_rule,basis
24,first_party_vehicle_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""車種"", ""2025"": ""車種""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks
25,weather_code_raw,string,原字串：事故天候代碼,not applicable,"{""2024"": ""天候"", ""2025"": ""天候""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 天候；例 6 雨、7 陰、8 晴
26,weather_code,string,事故天候代碼,類別碼字串，無數值大小尺度,"{""2024"": ""天候"", ""2025"": ""天候""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 天候；例 6 雨、7 陰、8 晴
27,weather_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""天候"", ""2025"": ""天候""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks
28,road_lighting_2024_code_raw,string,原字串：僅2024來源道路照明設備；不認證與另一年語義等價,not applicable,"{""2024"": ""道路照明設備""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1；與 2025 光線是否同義、是否發生重新編碼尚未確認
29,road_lighting_2024_code_applicable,boolean,來源年度是否包含此欄；False為不適用，不是來源缺失,logical flag；True/False/NA依欄定義,"{""2024"": ""道路照明設備""}",事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
30,road_lighting_2024_code,string,僅2024來源道路照明設備；不認證與另一年語義等價,類別碼字串，無數值大小尺度,"{""2024"": ""道路照明設備""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1；與 2025 光線是否同義、是否發生重新編碼尚未確認
31,road_lighting_2024_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""道路照明設備""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks
32,light_condition_2025_code_raw,string,原字串：僅2025來源光線；不認證與另一年語義等價,not applicable,"{""2025"": ""光線""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1；與 2025 光線是否同義、是否發生重新編碼尚未確認
33,light_condition_2025_code_applicable,boolean,來源年度是否包含此欄；False為不適用，不是來源缺失,logical flag；True/False/NA依欄定義,"{""2025"": ""光線""}",事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格


,name,dtype,definition,unit,source,observation_level,missing_rule,basis
36,road_class_first_party_code_raw,string,原字串：事故道路類別代碼，表單註明第 1 當事者,not applicable,"{""2024"": ""道路類別"", ""2025"": ""道路類別""}",第一當事者所在道路；PDF⑥,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1；例 5 市區道路
37,road_class_first_party_code,string,事故道路類別代碼，表單註明第 1 當事者,類別碼字串，無數值大小尺度,"{""2024"": ""道路類別"", ""2025"": ""道路類別""}",第一當事者所在道路；PDF⑥,NA保留，不以0代填,PDF p.1；例 5 市區道路
38,road_class_first_party_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""道路類別"", ""2025"": ""道路類別""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks
39,speed_limit_source_raw,string,原字串：道路速限，不是當事人實際車速；0、502 等值不自行更正,not applicable,"{""2024"": ""速限-速度限制"", ""2025"": ""速限-速度限制""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 道路速限；CSV 單位及 0 的發布規則待官方確認
40,speed_limit_source,Float64,道路速限，不是當事人實際車速；0、502 等值不自行更正,來源速限；km/h工作解讀，非實際速度,"{""2024"": ""速限-速度限制"", ""2025"": ""速限-速度限制""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 道路速限；CSV 單位及 0 的發布規則待官方確認
41,road_shape_code_raw,string,原字串：事故道路型態代碼,not applicable,"{""2024"": ""道路型態"", ""2025"": ""道路型態""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1；例 03 三岔、04 四岔、14 直路
42,road_shape_code,string,事故道路型態代碼,類別碼字串，無數值大小尺度,"{""2024"": ""道路型態"", ""2025"": ""道路型態""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1；例 03 三岔、04 四岔、14 直路
43,road_shape_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""道路型態"", ""2025"": ""道路型態""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks
44,accident_position_code_raw,string,原字串：事故發生位置類別,not applicable,"{""2024"": ""事故位置"", ""2025"": ""事故位置""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1；01 路口內、02 路口附近、03 待轉區、04 停等區
45,accident_position_code,string,事故發生位置類別,類別碼字串，無數值大小尺度,"{""2024"": ""事故位置"", ""2025"": ""事故位置""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1；01 路口內、02 路口附近、03 待轉區、04 停等區


,name,dtype,definition,unit,source,observation_level,missing_rule,basis
48,surface_material_code,string,路面鋪裝代碼,類別碼字串，無數值大小尺度,"{""2024"": ""路面狀況1"", ""2025"": ""路面狀況1""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 路面狀況 (1)
49,surface_material_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""路面狀況1"", ""2025"": ""路面狀況1""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks
50,surface_condition_code_raw,string,原字串：路面狀態代碼,not applicable,"{""2024"": ""路面狀況2"", ""2025"": ""路面狀況2""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 路面狀況 (2)；5 乾燥
51,surface_condition_code,string,路面狀態代碼,類別碼字串，無數值大小尺度,"{""2024"": ""路面狀況2"", ""2025"": ""路面狀況2""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 路面狀況 (2)；5 乾燥
52,surface_condition_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""路面狀況2"", ""2025"": ""路面狀況2""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks
53,surface_defect_code_raw,string,原字串：路面缺陷代碼,not applicable,"{""2024"": ""路面狀況3"", ""2025"": ""路面狀況3""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 路面狀況 (3)；4 無缺陷
54,surface_defect_code,string,路面缺陷代碼,類別碼字串，無數值大小尺度,"{""2024"": ""路面狀況3"", ""2025"": ""路面狀況3""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 路面狀況 (3)；4 無缺陷
55,surface_defect_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""路面狀況3"", ""2025"": ""路面狀況3""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks
56,road_obstacle_code_raw,string,原字串：障礙物代碼,not applicable,"{""2024"": ""道路障礙1"", ""2025"": ""道路障礙1""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 道路障礙 (1)；5 無障礙物
57,road_obstacle_code,string,障礙物代碼,類別碼字串，無數值大小尺度,"{""2024"": ""道路障礙1"", ""2025"": ""道路障礙1""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 道路障礙 (1)；5 無障礙物


... (7 outputs left out)

,name,dtype,definition,unit,source,observation_level,missing_rule,basis
144,accident_date,string,ISO日期YYYY-MM-DD；臺灣當地日期,date,年月日原欄,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
145,accident_time,string,HH:MM；原資料分鐘精度，不虛構秒數,local minute,時分原欄,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
146,accident_datetime,string,ISO分鐘時間YYYY-MM-DDTHH:MM+08:00；Asia/Taipei研究慣例,local datetime,年月日時分原欄,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
147,day_of_week,Int64,星期一0至星期日6，不等同國定假日,weekday,合法日期,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
148,accident_type_valid,boolean,該欄獨立結構／印列碼有效性；不認證死亡時間窗或全部語義,logical flag；True/False/NA依欄定義,事故類型及型態,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
149,accident_position_valid,boolean,該欄獨立結構／印列碼有效性；不認證死亡時間窗或全部語義,logical flag；True/False/NA依欄定義,事故位置,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
150,injured_count_structural_valid,boolean,該欄獨立結構／印列碼有效性；不認證死亡時間窗或全部語義,logical flag；True/False/NA依欄定義,受傷人數,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
151,fatalities_source_count_structural_valid,boolean,該欄獨立結構／印列碼有效性；不認證死亡時間窗或全部語義,logical flag；True/False/NA依欄定義,死亡人數,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
152,fatalities_2_30d_count_structural_valid,boolean,該欄獨立結構／印列碼有效性；不認證死亡時間窗或全部語義,logical flag；True/False/NA依欄定義,2-30日死亡人數,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
153,speed_limit_valid,Float64,預設速限補充數值：有限非負整數且排除0及502，不是實際車速,來源速限；km/h工作解讀，完整發布規格未認證,速限-速度限制,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格


,name,dtype,definition,unit,source,observation_level,missing_rule,basis
156,first_party_age_valid_structural,Float64,僅有限非負整數年齡；0及>100仍保留但待審，不認證為有效自然人年齡,歲（來源年齡）,年齡,第一當事人,NA保留，不以0代填,Unit2.1明示工作規格
157,age_negative_flag,boolean,負年齡排除於結構有效年齡,logical flag；True/False/NA依欄定義,年齡,第一當事人,NA保留，不以0代填,Unit2.1明示工作規格
158,age_zero_review_flag,boolean,0歲僅待確認，不直接判錯,logical flag；True/False/NA依欄定義,年齡,第一當事人,NA保留，不以0代填,Unit2.1明示工作規格
159,age_above_100_review_flag,boolean,超100歲僅待確認，不截尾,logical flag；True/False/NA依欄定義,年齡,第一當事人,NA保留，不以0代填,Unit2.1明示工作規格
160,longitude_candidate,Float64,X之經度工作解讀；datum未正式認證，採EPSG4326工作慣例,degree candidate,座標-X,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
161,latitude_candidate,Float64,Y之緯度工作解讀；datum未正式認證，無CRS轉換或geometry,degree candidate,座標-Y,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
162,coordinate_parse_valid,boolean,X及Y皆可解析為有限數；不是里內或CRS認證,logical flag；True/False/NA依欄定義,座標-X／Y,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
163,coordinate_missing_flag,boolean,任一原座標空白；不補值,logical flag；True/False/NA依欄定義,座標-X／Y,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
164,coordinate_global_range_valid,boolean,候選經緯度全球度數範圍；不是臺北位置認證,logical flag；True/False/NA依欄定義,座標-X／Y,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
165,morning_peak,boolean,07:00含至09:00不含；True有效符合／False有效不符／NA不可判定,logical flag；True/False/NA依欄定義,各自時間／日期／型態／位置／受傷欄,事故層級／診斷,有效性False則特徵NA；不得當False,Unit2.1明示工作規格


,name,dtype,definition,unit,source,observation_level,missing_rule,basis
168,evening_peak_valid,boolean,此特徵獨立有效分母資格；與其他特徵缺失無關,logical flag；True/False/NA依欄定義,evening_peak之定義欄,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
169,late_night,boolean,22:00含或06:00之前；True有效符合／False有效不符／NA不可判定,logical flag；True/False/NA依欄定義,各自時間／日期／型態／位置／受傷欄,事故層級／診斷,有效性False則特徵NA；不得當False,Unit2.1明示工作規格
170,late_night_valid,boolean,此特徵獨立有效分母資格；與其他特徵缺失無關,logical flag；True/False/NA依欄定義,late_night之定義欄,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
171,weekend,boolean,星期六或日；非所有國定假日；True有效符合／False有效不符／NA不可判定,logical flag；True/False/NA依欄定義,各自時間／日期／型態／位置／受傷欄,事故層級／診斷,有效性False則特徵NA；不得當False,Unit2.1明示工作規格
172,weekend_valid,boolean,此特徵獨立有效分母資格；與其他特徵缺失無關,logical flag；True/False/NA依欄定義,weekend之定義欄,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
173,intersection,boolean,位置1–4：路口及附近／待轉／停等；True有效符合／False有效不符／NA不可判定,logical flag；True/False/NA依欄定義,各自時間／日期／型態／位置／受傷欄,事故層級／診斷,有效性False則特徵NA；不得當False,Unit2.1明示工作規格
174,intersection_valid,boolean,此特徵獨立有效分母資格；與其他特徵缺失無關,logical flag；True/False/NA依欄定義,intersection之定義欄,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
175,rear_end,boolean,型態13追撞；True有效符合／False有效不符／NA不可判定,logical flag；True/False/NA依欄定義,各自時間／日期／型態／位置／受傷欄,事故層級／診斷,有效性False則特徵NA；不得當False,Unit2.1明示工作規格
176,rear_end_valid,boolean,此特徵獨立有效分母資格；與其他特徵缺失無關,logical flag；True/False/NA依欄定義,rear_end之定義欄,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
177,side_collision,boolean,型態16側撞，不混同擦撞；True有效符合／False有效不符／NA不可判定,logical flag；True/False/NA依欄定義,各自時間／日期／型態／位置／受傷欄,事故層級／診斷,有效性False則特徵NA；不得當False,Unit2.1明示工作規格


,name,dtype,definition,unit,source,observation_level,missing_rule,basis
180,pedestrian_collision_valid,boolean,此特徵獨立有效分母資格；與其他特徵缺失無關,logical flag；True/False/NA依欄定義,pedestrian_collision之定義欄,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
181,multi_injury,boolean,事故受傷人數至少2；不是重傷程度；True有效符合／False有效不符／NA不可判定,logical flag；True/False/NA依欄定義,各自時間／日期／型態／位置／受傷欄,事故層級／診斷,有效性False則特徵NA；不得當False,Unit2.1明示工作規格
182,multi_injury_valid,boolean,此特徵獨立有效分母資格；與其他特徵缺失無關,logical flag；True/False/NA依欄定義,multi_injury之定義欄,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
183,intersection_strict,boolean,額外嚴格路口：位置1；有效分母同intersection；True有效符合／False有效不符／NA不可判定,logical flag；True/False/NA依欄定義,各自時間／日期／型態／位置／受傷欄,事故層級／診斷,有效性False則特徵NA；不得當False,Unit2.1明示工作規格
184,intersection_strict_valid,boolean,此特徵獨立有效分母資格；與其他特徵缺失無關,logical flag；True/False/NA依欄定義,intersection_strict之定義欄,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
185,conditional_casualty_inconsistency_flag,boolean,R20條件疑點：程度1且原死亡0，或程度5且2–30日死亡0，或程度2且受傷0；不判哪欄錯誤,logical flag；True/False/NA依欄定義,受傷程度／三個事故傷亡原欄,事故層級／診斷,相應欄不可判則NA；不因此改multi_injury分母,Unit2.1明示工作規格
186,r20_check_valid,boolean,R20對應欄可判讀；不是傷亡語義認證,logical flag；True/False/NA依欄定義,受傷程度及相應總量,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
187,candidate_event_key_duplicate_flag,boolean,候選時地座標鍵重複，包含空白；不合併或刪除，非官方事件ID,logical flag；True/False/NA依欄定義,原年月日時分／地點／X／Y,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
188,time_address_key_duplicate_flag,boolean,只以日時分及地址的非唯一性；不拿座標作跨來源驗證前提,logical flag；True/False/NA依欄定義,原年月日時分／地點,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格
189,raw_record_duplicate_flag,boolean,第一當事人完整原始列重複；標記而不刪除,logical flag；True/False/NA依欄定義,全部適用原欄,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格


In [7]:
def load_standardized(csv_path,contract):
    # All cells first read as strings: no inferred NA/ID/category conversion.
    f=pd.read_csv(csv_path,encoding=contract['csv']['encoding'],dtype='string',keep_default_na=False,na_values=[contract['csv']['missing_sentinel']])
    assert list(f)==[x['name'] for x in contract['columns']]
    for field in contract['columns']:
        name=field['name'];dtype=field['dtype'];v=f[name]
        if dtype=='boolean':
            assert v.dropna().isin(['True','False']).all(),name
            f[name]=v.map({'True':True,'False':False}).astype('boolean')
        elif dtype=='Int64':f[name]=pd.to_numeric(v,errors='raise').astype('Int64')
        elif dtype=='Float64':f[name]=v.map(lambda x:pd.NA if pd.isna(x) else float(x)).astype('Float64')
        else:assert dtype=='string'
    return f


## 6. 正式輸出與獨立讀回

只寫入兩個已明確命名的新檔，確認processed目錄包含性。重新執行時僅接受相同bytes，禁止靜默覆寫不同版本。每筆資料保留、每欄dtype一致，原字串逐值保持。

In [8]:
OUTPUTS={'accidents':'data/processed/taipei_crashes_2024_2025_standardized.csv','schema':'data/processed/taipei_crashes_2024_2025_standardized_schema.json'}
paths={k:project_file(v,'processed',must_exist=False) for k,v in OUTPUTS.items()}
assert len(set(paths.values()))==2 and not set(paths.values())&{project_file(p) for p in EXPECTED_INPUTS}
csv_bytes=df.to_csv(index=False,na_rep='__MISSING__',lineterminator=chr(10)).encode('utf-8-sig')
schema_bytes=(json.dumps(contract,ensure_ascii=False,indent=2,sort_keys=False)+chr(10)).encode('utf-8')
for key,body in [('accidents',csv_bytes),('schema',schema_bytes)]:
    path=paths[key]
    if path.exists():assert path.read_bytes()==body,'Existing output differs; review version before overwriting: '+str(path.name)
    else:
        with path.open('xb') as stream:stream.write(body)
saved_contract=json.loads(paths['schema'].read_text(encoding='utf-8'))
reread=load_standardized(paths['accidents'],saved_contract)
pd.testing.assert_frame_equal(df,reread,check_dtype=True,check_exact=True)
assert reread.record_id.is_unique and len(reread)==45130
assert reread.groupby('source_year').size().to_dict()=={2024:22368,2025:22762}
assert reread.coordinate_missing_flag.sum()==3
assert not reread.duplicated(['source_file_sha256','source_row_number']).any()
for relative,expected in EXPECTED_INPUTS.items():assert sha(project_file(relative))==expected,relative
verification={key:dict(path=OUTPUTS[key],bytes=p.stat().st_size,sha256=sha(p),status='saved and independently read-back verified') for key,p in paths.items()}
summary={'status':'passed','rows':len(reread),'columns':len(reread.columns),'year_counts':{str(k):int(v) for k,v in reread.groupby('source_year').size().items()},'features':feature_counts,'quality':quality,'outputs':verification,'source_hashes_unchanged':True,'all_raw_values_traceable':True,'all_saved_values_and_dtypes_equal':True,'formal_accident_dataset_count':1,'spatial_assignment_performed':False,'human_review':'待DA驗收及PM審查；Codex本地協助'}
(TEMP/'processing_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
show('正式輸出身分及驗證',pd.DataFrame(verification).T)
print('All 16 requested categories of checks passed; one formal accident dataset, no village assignment.')


**正式輸出身分及驗證**

,path,bytes,sha256,status
accidents,data/processed/taipei_crashes_2024_2025_standardized.csv,50400834,b2ac0704068319b74f9aef2a28ad810d0cef473894a2a5cd23976f3fa7d2c252,saved and independently read-back verified
schema,data/processed/taipei_crashes_2024_2025_standardized_schema.json,230186,c3f064b3572d940489bdc86bb5c49dfaea8a228a3e7addbc8f9544bad324b650,saved and independently read-back verified


All 16 requested categories of checks passed; one formal accident dataset, no village assignment.


## 7. DE-to-DA交付與停止

本CSV已具統一欄位、逐欄有效值／原值、三態事故特徵及完整來源追溯。Unit2.2須重核正式資料hash後才建立固定里及空間歸屬；Unit2.3以各feature_valid決定有效分母，不能把NA當False。主要／敏感樣本候選44,212／45,083須由2.2重新驗證，2.1未排除任何空間異常。原死亡時間窗、R20、跨來源30／9受傷差異、datum與無官方事件ID限制保留。此為技術交付檢查，DA／PM人工驗收尚未完成。完成後停止，不自動執行Unit2.2。